# IDRA Capstone Project
## Optimizing Supply Chain Logistics: Inventory Management and Demand Forecasting
This notebook is designed to run from top to bottom and reproduces the capstone analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('P_3_supply_chain_dataset1.csv')
df.head()

## 1. Dataset Understanding

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all').T)
print('Missing values:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())

## 2. Data Cleaning

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.drop_duplicates().copy()

# Validate missing values and constant columns
print('Missing values after cleaning:\n', df.isna().sum())
print('Unique values in Stockout_Flag:', df['Stockout_Flag'].unique())

## 3. Feature Engineering and Preprocessing

In [ ]:
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter
df['Inventory_Gap'] = df['Inventory_Level'] - df['Reorder_Point']
df['Price_Margin'] = df['Unit_Price'] - df['Unit_Cost']
df['Markup_Ratio'] = df['Price_Margin'] / df['Unit_Cost']

target = 'Units_Sold'
features = ['SKU_ID','Warehouse_ID','Supplier_ID','Region','Inventory_Level',
            'Supplier_Lead_Time_Days','Reorder_Point','Order_Quantity',
            'Unit_Cost','Unit_Price','Promotion_Flag','Demand_Forecast',
            'Month','DayOfWeek','Quarter','Inventory_Gap','Price_Margin','Markup_Ratio']

categorical_features = [c for c in features if df[c].dtype == 'object']
numerical_features = [c for c in features if c not in categorical_features]

## 4. Exploratory Data Analysis

In [ ]:
display(df.groupby('Region')['Units_Sold'].agg(['count','mean','median','std']))
display(df.groupby('Promotion_Flag')['Units_Sold'].agg(['count','mean']))
display(df[numerical_features + [target]].corr(numeric_only=True)[target].sort_values(ascending=False))

plt.figure(figsize=(7,4))
plt.hist(df['Units_Sold'], bins=30)
plt.xlabel('Units Sold'); plt.ylabel('Frequency'); plt.title('Distribution of Units Sold')
plt.show()

plt.figure(figsize=(7,4))
df.groupby('Promotion_Flag')['Units_Sold'].mean().plot(kind='bar')
plt.ylabel('Average Units Sold'); plt.title('Average Demand by Promotion Status')
plt.show()

## 5. Machine Learning Model

In [ ]:
# Chronological split: earlier dates for training, later dates for testing
cutoff = pd.Timestamp('2024-10-18')
train = df[df['Date'] <= cutoff]
test = df[df['Date'] > cutoff]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numerical_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

model.fit(X_train, y_train)
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

results = pd.DataFrame({
    'Metric':['MAE','MSE','RMSE','R²'],
    'Training':[mean_absolute_error(y_train,train_pred),
                mean_squared_error(y_train,train_pred),
                mean_squared_error(y_train,train_pred)**0.5,
                r2_score(y_train,train_pred)],
    'Testing':[mean_absolute_error(y_test,test_pred),
               mean_squared_error(y_test,test_pred),
               mean_squared_error(y_test,test_pred)**0.5,
               r2_score(y_test,test_pred)]
})
display(results)

## 6. Save Supporting Dataset

In [ ]:
df.to_csv('Sharma_Pratyush_Capstone_Cleaned_Preprocessed.csv', index=False)
print('Saved cleaned/preprocessed dataset.')